# Test Train TinyLlama HelpSteer2 Adapters

This notebook is a test variant of Notebook 2. It uses one combined background training block and lets you set the number of selected training examples per attribute and the eval-loss check interval before starting the run.


## 1. Clone or update the repository

In [ ]:
%cd /content
import os
import shutil

repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"

if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

## 2. Check the GPU

In [ ]:
!nvidia-smi

## 3. Install dependencies

The pandas and NumPy versions are pinned for compatibility with the standard Colab environment. ArmoRM declares Transformers 4.40.0 and its custom model code relies on that version's internal Llama API, so Transformers, PEFT, and Accelerate are pinned to a compatible set. TinyLlama uses standard LoRA training without TorchAO or 4-bit quantization. Colab may include an old optional `torchao` package, so the install cell removes it.

In [ ]:
!pip uninstall -y torchao
!pip install -q -U "pandas==2.2.2" "numpy<2.1" "protobuf>=5.29.1,<6.0.0" "tensorboard==2.20.0" "transformers==4.40.0" "peft==0.10.0" "accelerate==0.29.3" datasets pyyaml

Restart the runtime once after installation so Python forgets previously imported Transformers, TorchAO, or PyTorch modules. This restart is required when replacing a newer Transformers version. Then rerun the repository cell and continue with the GPU and validation cells; you do not need to reinstall the dependencies again in the same Colab session.

## 4. Show important files

In [ ]:
!pwd
!ls
!ls configs
!ls scripts
!ls src

## 5. Test settings

Change these values before starting training. `TRAIN_EXAMPLES_PER_ATTRIBUTE` controls how many low-overlap, high-rated training texts are used for each adapter. `EVAL_LOSS_CHECK_STEPS` controls how often `eval_loss` is computed and logged. `LR_SCHEDULER_TYPE` can be `"cosine_decay"` or `"epoch_decay"`. `EPOCH_DECAY_FACTOR` controls the per-epoch learning-rate multiplier when `LR_SCHEDULER_TYPE = "epoch_decay"`. `SAVE_BEST_BY_EVAL_LOSS` controls whether the final saved adapter is the best held-out eval-loss checkpoint.


In [ ]:
TRAIN_EXAMPLES_PER_ATTRIBUTE = 8434
EVAL_LOSS_CHECK_STEPS = 100
LR_SCHEDULER_TYPE = "cosine_decay"
EPOCH_DECAY_FACTOR = 0.9
SAVE_BEST_BY_EVAL_LOSS = True

if TRAIN_EXAMPLES_PER_ATTRIBUTE < 1:
    raise ValueError("TRAIN_EXAMPLES_PER_ATTRIBUTE must be at least 1.")
if EVAL_LOSS_CHECK_STEPS < 1:
    raise ValueError("EVAL_LOSS_CHECK_STEPS must be at least 1.")
if LR_SCHEDULER_TYPE not in {"cosine_decay", "epoch_decay"}:
    raise ValueError("LR_SCHEDULER_TYPE must be cosine_decay or epoch_decay.")
if not 0 < EPOCH_DECAY_FACTOR <= 1:
    raise ValueError("EPOCH_DECAY_FACTOR must be greater than 0 and at most 1.")
if not isinstance(SAVE_BEST_BY_EVAL_LOSS, bool):
    raise ValueError("SAVE_BEST_BY_EVAL_LOSS must be True or False.")

print(f"Training examples per attribute: {TRAIN_EXAMPLES_PER_ATTRIBUTE}")
print(f"Eval loss check every {EVAL_LOSS_CHECK_STEPS} steps")
print(f"LR scheduler: {LR_SCHEDULER_TYPE}")
print(f"Epoch decay factor: {EPOCH_DECAY_FACTOR}")
print(f"Save best by eval loss: {SAVE_BEST_BY_EVAL_LOSS}")


## 6. Validate the config and inspect HelpSteer2

In [ ]:
!python scripts/validate_tinyllama_helpsteer2_config.py
!python scripts/inspect_helpsteer2_dataset.py --split "train" --max_examples {TRAIN_EXAMPLES_PER_ATTRIBUTE}

## 7. Start TensorBoard

Open the **Scalars** view to inspect training and evaluation loss against `global_step`. New points appear as training writes event logs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/tensorboard/tinyllama_helpsteer2

## 8. All adapters in one training block

The code block below starts one background script run that trains all five adapters sequentially: `complexity`, `verbosity`, `correctness`, `helpfulness`, and `coherence`.

Training reads the official HelpSteer2 `train` split and then selects low-overlap, high-rated examples per attribute set in `TRAIN_EXAMPLES_PER_ATTRIBUTE`. For each attribute, selection first exhausts stronger rating buckets (4, then 3, then lower allowed ratings); inside each rating bucket it prefers rows used fewer times before reusing overlap. Selection is planned globally in the order `complexity -> verbosity -> correctness -> helpfulness -> coherence`, while evaluation uses the separate official `validation` split. The first line removes stale stop files from an earlier run.

LoRA rank is fixed at `r=8` in the script, LoRA dropout is fixed at `0.1`, and `weight_decay=0.01` provides mild AdamW regularization. The first 6% of optimizer steps linearly warm the learning rate from near zero to `1e-4`. After warmup, `LR_SCHEDULER_TYPE` controls whether the learning rate follows smooth `cosine_decay` or stepwise `epoch_decay`. For `epoch_decay`, `EPOCH_DECAY_FACTOR` multiplies the learning rate once per epoch, while `--min_lr_ratio 0.1` keeps the floor at `0.1` of the initial learning rate (`1e-5`). Gradient clipping and gradient accumulation are fixed in the script; the pre-clipping `grad_norm` is logged to CSV and TensorBoard; precision is explicitly fixed to `bf16`; best-by-eval-loss saving is controlled from the settings cell above.

Training uses response-only SFT labels: prompt tokens up to and including the `Assistant:` marker are masked with `-100`, so `train_loss` and `eval_loss` are computed only on assistant response tokens. These loss curves are not directly comparable to older full prompt-response loss runs.

ArmoRM monitoring is temporarily disabled in the training command below. To enable it later, add `--use_armorm_monitoring` and the reward-monitoring arguments again.


In [ ]:
!rm -f STOP_CURRENT_ADAPTER STOP_TRAINING
!nohup python -u scripts/train_tinyllama_helpsteer2_adapters.py \
  --attributes complexity verbosity correctness helpfulness coherence \
  --split "train" \
  --eval_split "validation" \
  --max_training_examples {TRAIN_EXAMPLES_PER_ATTRIBUTE} \
  --num_epochs 5 \
  --batch_size 8 \
  --max_length 1024 \
  --learning_rate 1e-4 \
  --lr_scheduler_type {LR_SCHEDULER_TYPE} \
  --min_lr_ratio 0.1 \
  --epoch_decay_factor {EPOCH_DECAY_FACTOR} \
  --warmup_ratio 0.06 \
  --weight_decay 0.01 \
  --logging_steps 10 \
  --eval_steps {EVAL_LOSS_CHECK_STEPS} \
  --save_steps 500 \
  --save_best_by_eval_loss {SAVE_BEST_BY_EVAL_LOSS} \
  --use_tensorboard > /content/tinyllama_helpsteer2_training.log 2>&1 &

## 9. Stop the current adapter

Run this cell while background training is active if you want to stop the current adapter. The script does not wait until the end of the epoch. It stops after the next save step, saves the adapter, removes `STOP_CURRENT_ADAPTER`, and exits gracefully. With `--save_steps 500`, the maximum wait is roughly until the next 500-step checkpoint.

In [ ]:
!touch STOP_CURRENT_ADAPTER

## 10. Stop all training

Run this cell while background training is active if you want to stop the full training run. The script stops after the next save step, saves the currently running adapter, and exits gracefully. With `--save_steps 500`, the maximum wait is roughly until the next 500-step checkpoint.

In [ ]:
!touch STOP_TRAINING

## 11. Check the running training
Run these cells at any time to see whether an adapter process is active and to inspect the latest training output. The adapter commands write to `/content/tinyllama_helpsteer2_training.log`.

In [ ]:
!pgrep -af "[t]rain_tinyllama_helpsteer2_adapters.py" || echo "No adapter training process is running."
!latest_epoch=$(grep -E "Epoch [0-9]+/[0-9]+.*shuffle_seed=" /content/tinyllama_helpsteer2_training.log 2>/dev/null | tail -n 1); [ -n "$latest_epoch" ] && echo "$latest_epoch" || echo "No epoch seed found yet."
!tail -n 30 /content/tinyllama_helpsteer2_training.log 2>/dev/null || echo "No training log exists yet."

In [ ]:
!pgrep -af "[t]rain_tinyllama_helpsteer2_adapters.py" || echo "No adapter training process is running."
!grep -i "eval_loss" /content/tinyllama_helpsteer2_training.log 2>/dev/null | tail -n 30 || echo "No eval_loss found yet."

In [ ]:
!pgrep -af "[t]rain_tinyllama_helpsteer2_adapters.py" || echo "No adapter training process is running."
!grep "objective=helpsteer-helpfulness" /content/tinyllama_helpsteer2_training.log 2>/dev/null | tail -n 30 || echo "No helpsteer-helpfulness ArmoRM monitoring found yet."

Check the ArmoRM download cache if training is currently loading the reward monitor.

In [ ]:
# Check ArmoRM Download
!du -sh /root/.cache/huggingface/hub/models--RLHFlow--ArmoRM-Llama3-8B-v0.1 2>/dev/null || echo "ArmoRM cache not created yet."
!ls -lh /root/.cache/huggingface/hub/models--RLHFlow--ArmoRM-Llama3-8B-v0.1/snapshots/*/ 2>/dev/null || echo "ArmoRM snapshot not created yet."

## 12. Inspect saved logs
Use these folders to inspect the CSV logs and TensorBoard event files.

In [ ]:
!ls results/tinyllama_helpsteer2_training_logs
!ls results/tensorboard/tinyllama_helpsteer2

## 13. Export training curves

In [ ]:
!python scripts/export_tinyllama_training_curves.py || true

## 14. Final adapter check

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py

## 15. Create a local adapter backup
Run this only after all five adapters are finished. The zip file is a generated local backup and must stay out of Git.

In [ ]:
!zip -r tinyllama_helpsteer2_adapters.zip adapters/tinyllama-helpsteer2-*-adapter/

## 16. Git safety check
Adapters, checkpoints, safetensors, `.bin` files, model weights, TensorBoard events, and zip files are generated artifacts. Keep them out of Git unless a small result file is intentionally selected later.

In [ ]:
!git status